In [ ]:
# Experiment 3: Logistic Regression hyperparameter tuning with GridSearchCV

# Import necessary libraries
import mlflow
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np
import os
import dagshub


In [ ]:
# Set MLflow tracking URI to DagsHub
mlflow.set_tracking_uri("https://dagshub.com/DeepuML/Mlops-Mini-Project.mlflow")

Accessing as DeepuML

Initialized MLflow to track repo "DeepuML/Mlops-Mini-Project"

Repository DeepuML/Mlops-Mini-Project initialized!

In [5]:
df = pd.read_csv('https://raw.githubusercontent.com/campusx-official/jupyter-masterclass/main/tweet_emotions.csv').drop(columns=['tweet_id'])
df

,sentiment,content
0,empty,@tiffanylue i know i was listenin to bad habi...
1,sadness,Layin n bed with a headache ughhhh...waitin o...
2,sadness,Funeral ceremony...gloomy friday...
3,enthusiasm,wants to hang out with friends SOON!
4,neutral,@dannycastillo We want to trade with someone w...
...,...,...
39995,neutral,@JohnLloydTaylor
39996,love,Happy Mothers Day All my love
39997,love,Happy Mother's Day to all the mommies out ther...
39998,happiness,@niariley WASSUP BEAUTIFUL!!! FOLLOW ME!! PEE...


In [6]:

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['content'] = df['content'].apply(lower_case)
        df['content'] = df['content'].apply(remove_stop_words)
        df['content'] = df['content'].apply(removing_numbers)
        df['content'] = df['content'].apply(removing_punctuations)
        df['content'] = df['content'].apply(removing_urls)
        df['content'] = df['content'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

<>:30: SyntaxWarning: invalid escape sequence '\s'
<>:30: SyntaxWarning: invalid escape sequence '\s'
C:\Users\Deepu\AppData\Local\Temp\ipykernel_20504\3431972446.py:30: SyntaxWarning: invalid escape sequence '\s'
  text = re.sub('\s+', ' ', text).strip()


In [15]:
# Reload and process the data properly
df = pd.read_csv('https://raw.githubusercontent.com/campusx-official/jupyter-masterclass/main/tweet_emotions.csv').drop(columns=['tweet_id'])

# Filter for happiness and sadness first
x = df['sentiment'].isin(['happiness','sadness'])
df = df[x]
print(f"After filtering: {df.shape[0]} samples")

# Normalize the text data
df = normalize_text(df)

# Convert sentiment to binary
df['sentiment'] = df['sentiment'].replace({'sadness':0, 'happiness':1})

print(f"Final data shape: {df.shape}")
print(f"Sentiment distribution:\n{df['sentiment'].value_counts()}")
print(f"Sample of processed text:\n{df['content'].head(3).tolist()}")

After filtering: 10374 samples
Final data shape: (10374, 2)
Sentiment distribution:
sentiment
1    5209
0    5165
Name: count, dtype: int64
Sample of processed text:
['layin n bed headache ughhhh waitin call', 'funeral ceremony gloomy friday', 'sleep im not thinking old friend want married now damn amp want scandalous']
Final data shape: (10374, 2)
Sentiment distribution:
sentiment
1    5209
0    5165
Name: count, dtype: int64
Sample of processed text:
['layin n bed headache ughhhh waitin call', 'funeral ceremony gloomy friday', 'sleep im not thinking old friend want married now damn amp want scandalous']


C:\Users\Deepu\AppData\Local\Temp\ipykernel_20504\2663186393.py:13: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['sentiment'] = df['sentiment'].replace({'sadness':0, 'happiness':1})


In [13]:
# Check the preprocessed data
print("Data shape:", df.shape)
print("\nFirst few preprocessed texts:")
print(df['content'].head(10).tolist())
print("\nCheck for empty strings:")
print("Empty strings count:", (df['content'] == '').sum())
print("Very short strings (< 2 chars):", (df['content'].str.len() < 2).sum())

Data shape: (0, 2)

First few preprocessed texts:
[]

Check for empty strings:
Empty strings count: 0
Very short strings (< 2 chars): 0


In [14]:
# Let's reload and check the original data
df_original = pd.read_csv('https://raw.githubusercontent.com/campusx-official/jupyter-masterclass/main/tweet_emotions.csv').drop(columns=['tweet_id'])
print("Original data shape:", df_original.shape)
print("\nSentiment value counts:")
print(df_original['sentiment'].value_counts())
print("\nUnique sentiment values:")
print(df_original['sentiment'].unique())

Original data shape: (40000, 2)

Sentiment value counts:
sentiment
neutral       8638
worry         8459
happiness     5209
sadness       5165
love          3842
surprise      2187
fun           1776
relief        1526
hate          1323
empty          827
enthusiasm     759
boredom        179
anger          110
Name: count, dtype: int64

Unique sentiment values:
['empty' 'sadness' 'enthusiasm' 'neutral' 'worry' 'surprise' 'love' 'fun'
 'hate' 'happiness' 'boredom' 'relief' 'anger']


In [16]:
# Create features and split data
vectorizer = CountVectorizer(max_features=1000)
X = vectorizer.fit_transform(df['content'])
y = df['sentiment']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Feature matrix shape: {X.shape}")
print(f"Training set: {X_train.shape}, Test set: {X_test.shape}")
print(f"Target distribution - Train: {np.bincount(y_train)}, Test: {np.bincount(y_test)}")

Feature matrix shape: (10374, 1000)
Training set: (8299, 1000), Test set: (2075, 1000)
Target distribution - Train: [4105 4194], Test: [1060 1015]


In [ ]:
# Set the experiment name
mlflow.set_experiment("LoR Hyperparameter Tuning")

2025/09/28 01:05:27 INFO mlflow.tracking.fluent: Experiment with name 'LoR Hyperparameter Tuning' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/ecc7612053274a989e7d96ec6d427075', creation_time=1759001727994, experiment_id='2', last_update_time=1759001727994, lifecycle_stage='active', name='LoR Hyperparameter Tuning', tags={}>

In [ ]:
import pickle
import tempfile
import warnings
from urllib3.exceptions import InsecureRequestWarning

# Suppress warnings
warnings.filterwarnings('ignore', category=InsecureRequestWarning)
os.environ["MLFLOW_TRACKING_INSECURE_TLS"] = "true"

# Define hyperparameters to tune
param_grid = {
    'C': [0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear', 'saga'],
    'max_iter': [100, 500, 1000]
}

# Perform Grid Search
print("Starting hyperparameter tuning...")
grid_search = GridSearchCV(
    LogisticRegression(random_state=42),
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

# Fit the grid search
grid_search.fit(X_train, y_train)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best cross-validation F1 score: {grid_search.best_score_:.4f}")

# Get the best model
best_model = grid_search.best_estimator_

# Evaluate on test set
y_pred = best_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("\n" + "="*50)
print("BEST MODEL EVALUATION RESULTS")
print("="*50)
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print("="*50)

In [ ]:
# Log the best model and results to MLflow
with mlflow.start_run(run_name="Best Logistic Regression - Hyperparameter Tuning"):
    
    # Log preprocessing parameters
    mlflow.log_param("vectorizer", "CountVectorizer")
    mlflow.log_param("max_features", 1000)
    mlflow.log_param("test_size", 0.2)
    
    # Log best hyperparameters
    for param, value in grid_search.best_params_.items():
        mlflow.log_param(f"best_{param}", value)
    
    # Log cross-validation score
    mlflow.log_metric("cv_f1_score", grid_search.best_score_)
    
    # Log evaluation metrics
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)
    
    # Save and log the best model manually for DagsHub compatibility
    with tempfile.NamedTemporaryFile(suffix='.pkl', delete=False) as f:
        pickle.dump(best_model, f)
        model_path = f.name
    
    mlflow.log_artifact(model_path, "model")
    
    # Also save vectorizer
    with tempfile.NamedTemporaryFile(suffix='.pkl', delete=False) as f:
        pickle.dump(vectorizer, f)
        vectorizer_path = f.name
    
    mlflow.log_artifact(vectorizer_path, "model")
    
    # Clean up temporary files
    os.unlink(model_path)
    os.unlink(vectorizer_path)
    
    # Get current run info
    current_run = mlflow.active_run()
    run_id = current_run.info.run_id
    
    print("\n" + "="*60)
    print(" HYPERPARAMETER TUNING COMPLETED & LOGGED TO MLFLOW")
    print("="*60)
    print(f" Run ID: {run_id}")
    print(f" Best F1 Score: {f1:.4f}")
    print(f" Best Parameters: {grid_search.best_params_}")
    print("="*60)

In [18]:
# Define hyperparameter grid for Logistic Regression
param_grid = {
    'C': [0.1, 1, 10],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear']
}


In [21]:
import pickle
import tempfile
import warnings
from urllib3.exceptions import InsecureRequestWarning

# Suppress warnings
warnings.filterwarnings('ignore', category=InsecureRequestWarning)
os.environ["MLFLOW_TRACKING_INSECURE_TLS"] = "true"

# Start the parent run for hyperparameter tuning
with mlflow.start_run(run_name="Logistic Regression Hyperparameter Tuning"):

    # Perform grid search
    grid_search = GridSearchCV(LogisticRegression(), param_grid, cv=5, scoring='f1', n_jobs=-1)
    grid_search.fit(X_train, y_train)

    # Log each parameter combination as a child run
    for params, mean_score, std_score in zip(grid_search.cv_results_['params'], 
                                            grid_search.cv_results_['mean_test_score'], 
                                            grid_search.cv_results_['std_test_score']):
        with mlflow.start_run(run_name=f"LR_C={params['C']}_penalty={params['penalty']}", nested=True):
            model = LogisticRegression(**params)
            model.fit(X_train, y_train)
            
            # Model evaluation
            y_pred = model.predict(X_test)
            accuracy = accuracy_score(y_test, y_pred)
            precision = precision_score(y_test, y_pred)
            recall = recall_score(y_test, y_pred)
            f1 = f1_score(y_test, y_pred)
            
            # Log parameters and metrics
            mlflow.log_params(params)
            mlflow.log_metric("mean_cv_score", mean_score)
            mlflow.log_metric("std_cv_score", std_score)
            mlflow.log_metric("accuracy", accuracy)
            mlflow.log_metric("precision", precision)
            mlflow.log_metric("recall", recall)
            mlflow.log_metric("f1_score", f1)
            
            # Save model manually for DagsHub compatibility
            with tempfile.NamedTemporaryFile(suffix='.pkl', delete=False) as f:
                pickle.dump(model, f)
                model_path = f.name
            
            mlflow.log_artifact(model_path, "model")
            os.unlink(model_path)  # Clean up
            
            # Print the results for verification
            print("="*50)
            print(f" Parameters: {params}")
            print(f" Mean CV Score: {mean_score:.4f}, Std: {std_score:.4f}")
            print(f" Test Accuracy: {accuracy:.4f}")
            print(f" Test F1 Score: {f1:.4f}")
            print("="*50)

    # Get the best model and evaluate it
    best_model = grid_search.best_estimator_
    best_y_pred = best_model.predict(X_test)
    best_accuracy = accuracy_score(y_test, best_y_pred)
    best_f1 = f1_score(y_test, best_y_pred)
    
    # Log the best run details in the parent run
    best_params = grid_search.best_params_
    best_score = grid_search.best_score_
    mlflow.log_params(best_params)
    mlflow.log_metric("best_cv_f1_score", best_score)
    mlflow.log_metric("best_test_accuracy", best_accuracy)
    mlflow.log_metric("best_test_f1_score", best_f1)
    
    # Save the best model
    with tempfile.NamedTemporaryFile(suffix='.pkl', delete=False) as f:
        pickle.dump(best_model, f)
        best_model_path = f.name
    
    mlflow.log_artifact(best_model_path, "best_model")
    
    # Also save vectorizer
    with tempfile.NamedTemporaryFile(suffix='.pkl', delete=False) as f:
        pickle.dump(vectorizer, f)
        vectorizer_path = f.name
    
    mlflow.log_artifact(vectorizer_path, "best_model")
    
    # Clean up
    os.unlink(best_model_path)
    os.unlink(vectorizer_path)
    
    print("\n" + "="*60)
    print(" HYPERPARAMETER TUNING COMPLETED")
    print("="*60)
    print(f" Best Parameters: {best_params}")
    print(f" Best CV F1 Score: {best_score:.4f}")
    print(f" Best Test F1 Score: {best_f1:.4f}")
    print(f" Best Test Accuracy: {best_accuracy:.4f}")
    print("="*60)

 Parameters: {'C': 0.1, 'penalty': 'l1', 'solver': 'liblinear'}
 Mean CV Score: 0.7052, Std: 0.0142
 Test Accuracy: 0.7398
 Test F1 Score: 0.7125
🏃 View run LR_C=0.1_penalty=l1 at: https://dagshub.com/DeepuML/Mlops-Mini-Project.mlflow/#/experiments/2/runs/2c218317f7d44d5cba2ca5715ee8b32b
🧪 View experiment at: https://dagshub.com/DeepuML/Mlops-Mini-Project.mlflow/#/experiments/2
🏃 View run LR_C=0.1_penalty=l1 at: https://dagshub.com/DeepuML/Mlops-Mini-Project.mlflow/#/experiments/2/runs/2c218317f7d44d5cba2ca5715ee8b32b
🧪 View experiment at: https://dagshub.com/DeepuML/Mlops-Mini-Project.mlflow/#/experiments/2
 Parameters: {'C': 0.1, 'penalty': 'l2', 'solver': 'liblinear'}
 Mean CV Score: 0.7750, Std: 0.0113
 Test Accuracy: 0.7807
 Test F1 Score: 0.7766
 Parameters: {'C': 0.1, 'penalty': 'l2', 'solver': 'liblinear'}
 Mean CV Score: 0.7750, Std: 0.0113
 Test Accuracy: 0.7807
 Test F1 Score: 0.7766
🏃 View run LR_C=0.1_penalty=l2 at: https://dagshub.com/DeepuML/Mlops-Mini-Project.mlflow/#/e